# AL8 — Transfer Learning: a Trained ResNet on 1958 Air Photos
**MCS 3950 Computer Vision · UPEI · Fall 2026 · Friday Nov 13**

In **AL7** you trained a CNN from scratch on 10,000 labelled photos. In **A2** you are training one
on just **630** labelled 1958 air-photo tiles. In **L14** you saw the alternative: start from a
ResNet-18 that has already learned from 1.2 million ImageNet photos, and reuse it.

Today you do that on **your own kind of data**: the A2 tiles (woodland, field, edge, built), with
**Kings County** kept back as a place the network never sees. It is the recipe you will use in your
project:

1. **Get the tiles into the network's format** (grey → 3 channels, resize, ImageNet normalisation),
   and use the frozen network as a **feature extractor** with a new last layer.
2. **Fine-tune**: let every weight move, gently.
3. **Augment** with changes that keep an air photo's label true.
4. Test **once**, on Kings County.

| Part | Topic | Time |
|------|-------|------|
| 0 | Setup: the tiles and the trained network | ~5 min |
| 1 | Your tiles in an ImageNet network; feature extraction | ~13 min |
| 2 | Fine-tuning, and how big a step | ~12 min |
| 3 | Augmentation for air photos | ~11 min |
| 4 | Kings County, once | ~6 min |
| ★ | Bonus: test-time augmentation | if time |

There are **three tasks** and a few **checkpoint questions**. Before each experiment there is a
🔮 **Predict first** prompt: write your guess down *before* you run the cell.

> **Use a GPU.** In Colab: **Runtime → Change runtime type → T4 GPU**, then run the setup cell.
> With a GPU the whole lab trains in about a minute; on a CPU it takes about 5 minutes of
> training in total (each fine-tuning run is about 2 minutes), so pair up if you can't get a GPU.
>
> **Not submitted or graded.** This is Test 3 material (L12–L15). It uses A2's data file, but
> none of A2's tasks: A2 trains a network **from scratch**; today nothing is trained from scratch.

In [ ]:
import io, os, time, copy, hashlib, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
GOLD, RED, TEAL, GREEN, NAVY, GREY = "#C8962C", "#B82828", "#1A7A8A", "#2E7D50", "#1A3A5C", "#9AA5B1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} on {DEVICE.upper()}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
if DEVICE == "cpu":
    torch.set_num_threads(max(2, torch.get_num_threads()))
    print("No GPU. Colab: Runtime -> Change runtime type -> T4 GPU, then re-run this cell.\n"
          "On a CPU, the training in this lab takes about 5 minutes in total.")

def n_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)

## Part 0 — The tiles and the trained network  `⏱ ~5 min`

`A2_tiles_1958.npz` (~4 MB) is the A2 data file: 64 × 64 grey tiles at 2.32 m per pixel (each tile
is 148 m on a side), cut from the 1958 county mosaics and labelled by hand.

| Split | Tiles | From | Used for |
|---|---|---|---|
| `train` | 630 | Prince + Queens | learning the weights |
| `val` | 157 | Prince + Queens | choosing settings |
| `test` | 260 | **Kings** | **once**, at the very end (Part 4) |

If you have already run A2 in this Colab session, the file is already here.

In [ ]:
BASE = ("https://raw.githubusercontent.com/andrewgodbout/"
        "MCS-3950-F26/main/Notebooks/data/")
FILE = "A2_tiles_1958.npz"
for path in (FILE, os.path.join("data", FILE)):
    if os.path.exists(path):
        print(f"{FILE}: already present"); break
else:
    with urllib.request.urlopen(BASE + FILE, timeout=180) as r:
        blob = r.read()
    with open(FILE, "wb") as f:
        f.write(blob)
    path = FILE
    print(f"{FILE}: {len(blob)/1e6:.1f} MB downloaded")

D = np.load(path)
CLASSES = [str(c) for c in D["classes"]]
X_train, X_val, X_test = D["X_train"], D["X_val"], D["X_test"]
y_train, y_val, y_test = (D[k].astype(int) for k in ("y_train", "y_val", "y_test"))
for name, y in (("train", y_train), ("val", y_val), ("test (Kings)", y_test)):
    print(f"{name:13s} {len(y):4d} tiles: " + ", ".join(f"{CLASSES[c]} {n}" for c, n in enumerate(np.bincount(y, minlength=4))))

### The trained network

The usual one line is

```python
model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
```

It downloads PyTorch's own ImageNet weights (45 MB) once. The cell below does exactly that, and
keeps a copy of the weights so that every model today can start from the same place. If the
download is blocked (some networks block it), it falls back to another trained copy of the same
network, hosted on GitHub: the one L14 used. Either works for everything below.

In [ ]:
try:
    STATE = torchvision.models.resnet18(weights="IMAGENET1K_V1").state_dict()
    WEIGHTS = "torchvision IMAGENET1K_V1"
except Exception as e:
    print("IMAGENET1K_V1 download failed:", type(e).__name__, "-> using the fallback weights")
    url = ("https://github.com/huggingface/pytorch-image-models/releases/download/"
           "v0.1-rsb-weights/resnet18_a1_0-d63eafa0.pth")
    STATE = torch.hub.load_state_dict_from_url(url, map_location="cpu", progress=False)
    WEIGHTS = "timm resnet18 a1 (fallback)"

def pretrained_resnet18():
    m = torchvision.models.resnet18(weights=None)        # the L13 architecture ...
    m.load_state_dict(STATE)                              # ... with ImageNet-trained weights
    return m

IMAGENET_CLASSES = torchvision.models.ResNet18_Weights.IMAGENET1K_V1.meta["categories"]
net = pretrained_resnet18()
print(f"weights: {WEIGHTS}")
print(f"ResNet-18: {n_params(net):,d} parameters; last layer fc: {net.fc.in_features} -> {net.fc.out_features} ImageNet classes")

---
## Part 1 — Your tiles in an ImageNet network  `⏱ ~13 min`

### Step 1 · From a grey 64 × 64 tile to the network's input  (Task 1)

ImageNet photos are **colour** and about **224 × 224**. The network was trained on them after
each colour channel was **normalised** with ImageNet's mean and standard deviation (L9's
centring again):

| channel | R | G | B |
|---|---|---|---|
| mean | 0.485 | 0.456 | 0.406 |
| std | 0.229 | 0.224 | 0.225 |

So that is the input it expects. Our tiles are grey, 64 × 64, with values 0–255. Four steps:

1. scale to 0–1 (divide by 255);
2. copy the grey channel into R, G and B: `(N, 64, 64)` → `(N, 3, 64, 64)`;
3. enlarge to `SIZE` × `SIZE` (112, as in L14: big enough for the network's five halvings,
   and a quarter of the cost of 224) with `F.interpolate(x, size=SIZE, mode="bilinear")`;
4. subtract each channel's mean and divide by its std.

**By hand first.** A **white** pixel (255) becomes 1.0 after step 1. What does it become in the
**R** channel after step 4? And a pixel of value 0?

**Task 1:** complete `prep`. It must accept either a NumPy `uint8` array or a float tensor (Part 3
will pass it augmented float tiles), always with values on the 0–255 scale.

In [ ]:
SIZE = 112
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def prep(tiles):
    '''
    tiles : (N, 64, 64) grey tiles, 0..255 (NumPy uint8 or a float tensor)
    returns (N, 3, SIZE, SIZE) float tensor on DEVICE, normalised with ImageNet's mean and std
    '''
    x = torch.as_tensor(tiles, device=DEVICE).float()       # (N, 64, 64), 0..255
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────────
    # normalize to 0-1
    #x = x / 255
    # copy to all channels RGB
    #x = x[:, None].repeat(1, 3, 1, 1)  #                   (N, 3, 64, 64)
    # resize
    #x = F.interpolate(x, size=SIZE, mode="bilinear", align_corners=False)
    # use the model's mean and std
    #return (x - IMNET_MEAN) / IMNET_STD
    # ────────────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Task 1: four lines.")

In [ ]:
# Check Task 1
try:
    out = prep(np.stack([np.full((64, 64), 255, np.uint8), np.zeros((64, 64), np.uint8)]))
    assert tuple(out.shape) == (2, 3, SIZE, SIZE), f"shape should be (2, 3, {SIZE}, {SIZE}), got {tuple(out.shape)}"
    white, black = out[0, :, 0, 0].cpu().numpy(), out[1, :, 0, 0].cpu().numpy()
    print("white pixel ->", white.round(3), "  black pixel ->", black.round(3))
    assert np.allclose(white, [2.249, 2.429, 2.640], atol=0.01), "white should give (1 - mean) / std per channel"
    assert np.allclose(black, [-2.118, -2.036, -1.804], atol=0.01), "black should give (0 - mean) / std per channel"
    assert torch.allclose(prep(torch.tensor(X_train[:2]).float()), prep(X_train[:2])), "float input should give the same answer"
    print("Task 1 OK")
except NotImplementedError:
    print("Complete Task 1 first.")

### Step 2 · 🔮 What does ImageNet think these are?

ImageNet has 1,000 classes: dog breeds, cars, mugs, mushrooms... and not one of them is
"woodland" or "field". **Predict first:** the unmodified network has never seen an air photo. What
will it call these tiles? And do you expect what it has learned to be **useful** for our four
classes anyway?

In [ ]:
try:
    show = np.concatenate([np.where(y_val == c)[0][:3] for c in range(4)])
    net = pretrained_resnet18().to(DEVICE).eval()
    with torch.no_grad():
        P = net(prep(X_val[show])).softmax(1).cpu()
    fig, axes = plt.subplots(2, 6, figsize=(16, 6.4), gridspec_kw={"hspace": 0.45})
    for k, p, ax in zip(show, P, axes.ravel()):
        top = p.topk(2)
        ax.imshow(X_val[k], cmap="gray", vmin=0, vmax=255); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"really: {CLASSES[y_val[k]]}", fontsize=11, color=NAVY)
        ax.set_xlabel("\n".join(f"{IMAGENET_CLASSES[j]} {v:.0%}" for v, j in zip(top.values, top.indices)),
                      fontsize=9.5, color=RED)
    plt.suptitle("The ImageNet network's top-2 guesses (red) for 12 validation tiles", y=0.99)
    plt.show()
except NotImplementedError:
    print("Complete Task 1 first.")

### Step 3 · The network as a feature extractor

Its **answers** are useless here, but L14 showed that the 512 numbers just **before** the last
layer (after global average pooling) describe an image's content: textures, edges, blobs. We cut
off the last layer, run every tile through the rest of the network (the **body**) **once**, and
keep those 512 numbers per tile. The body is frozen, so these never change.

In [ ]:
@torch.no_grad()
def features(tiles, model=None, bs=128, prep_fn=None):
    '''(N, 64, 64) tiles -> (N, 512) features from the frozen body (everything before fc).'''
    model = model or pretrained_resnet18().to(DEVICE)
    prep_fn = prep_fn or prep
    model.eval()
    #all but the last layer
    body = nn.Sequential(*list(model.children())[:-1])     # conv1 ... layer4, avgpool (drop fc)
    return torch.cat([body(prep_fn(tiles[i:i + bs])).flatten(1) for i in range(0, len(tiles), bs)])

try:
    t0 = time.time()

    # convert the images to features
    F_train, F_val, F_test = features(X_train), features(X_val), features(X_test)
    print(f"features: train {tuple(F_train.shape)}, val {tuple(F_val.shape)}, test {tuple(F_test.shape)}  ({time.time() - t0:.1f} s)")
except NotImplementedError:
    print("Complete Task 1 first.")

### Step 4 · A new last layer

The new last layer is `nn.Linear(512, 4)`: 4 class scores from 512 features. **By hand:** how
many weights does it have? (Rows × columns, plus one bias per output.)

`train_head` is the same small training loop as L14 §5: Adam, the whole training set as one batch,
300 steps, a little weight decay. On 512 numbers per tile instead of a whole network, it takes a
second or two even on a CPU.

In [ ]:
def train_head(f, y, steps=300, lr=0.01, wd=1e-3, seed=0):
    torch.manual_seed(seed)
    head = nn.Linear(512, 4).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=wd)
    y = torch.as_tensor(y, device=DEVICE)
    for _ in range(steps):
        loss = F.cross_entropy(head(f), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return head

@torch.no_grad()
def head_acc(head, f, y):
    return (head(f).argmax(1).cpu().numpy() == y).mean()

try:
    t0 = time.time()

    # train just the new last layer on the features
    HEAD = train_head(F_train, y_train)
    print(f"new last layer: {n_params(HEAD):,d} weights, trained in {time.time() - t0:.1f} s")
    print(f"feature extraction: train {head_acc(HEAD, F_train, y_train):.1%}   val {head_acc(HEAD, F_val, y_val):.1%}")
    RESULTS = {"feature extraction": {"val": head_acc(HEAD, F_val, y_val)}}
except NameError:
    print("Complete Task 1 (and Step 3) first.")

Compare that validation number with your **A2** network trained from scratch, if you have run it
(A2 Part 5.2 prints `val_acc`). Same tiles, same validation set. Fill it in if you have it; the lab
does not need it.

In [ ]:
MY_A2_VAL = None      # e.g. 0.78, your A2 TileCNN's val_acc (optional)
if MY_A2_VAL is not None and "RESULTS" in globals():
    print(f"A2, from scratch: {MY_A2_VAL:.1%}  |  feature extraction: {RESULTS['feature extraction']['val']:.1%}")

### Step 5 · 🔮 What if you forget the normalisation?

A common bug: the tiles go in on the **0–255** scale (no `/ 255`, no mean and std).
**Predict first:** better, worse, or about the same? By how much?

In [ ]:
def prep_raw(tiles):            # the bug: no /255 and no ImageNet mean/std
    x = torch.as_tensor(tiles, device=DEVICE).float()[:, None].repeat(1, 3, 1, 1)
    return F.interpolate(x, size=SIZE, mode="bilinear", align_corners=False)

try:
    HEAD                        # needs Task 1 and Step 4
    f_tr, f_va = features(X_train, prep_fn=prep_raw), features(X_val, prep_fn=prep_raw)
    h = train_head(f_tr, y_train)
    print(f"0..255, not normalised:   train {head_acc(h, f_tr, y_train):.1%}   val {head_acc(h, f_va, y_val):.1%}")
    print(f"ImageNet normalisation:   train {head_acc(HEAD, F_train, y_train):.1%}   val {head_acc(HEAD, F_val, y_val):.1%}")
    print(f"typical input value: raw {prep_raw(X_val[:50]).abs().mean():.0f}   normalised {prep(X_val[:50]).abs().mean():.2f}")
except NameError:
    print("Complete Task 1 (and Steps 3-4) first.")

### Checkpoint 1

**Q1a.** The ImageNet network called our tiles things like those in Step 2. Why are its
answers nonsense here, and why can its 512 features still be useful for woodland vs field vs edge
vs built? (Think about what the early layers detect, L14 §1.)

*Your answer:*

**Q1b.** (i) How many weights does the new last layer have? Show the arithmetic. What
fraction is that of the whole ResNet-18? (ii) Forgetting the normalisation hurt. Explain why, in
terms of what the first layer's filters were trained on.

*Your answer:*

---
## Part 2 — Fine-tuning  `⏱ ~12 min`

Feature extraction never changes the body. Its filters were tuned for ImageNet photos; **fine-
tuning** lets them adapt to air photos too. We put the whole network together: trained body + a new
last layer, and decide which parts may move.

### Step 1 · A model with a new head  (Task 2)

In PyTorch, a weight is frozen when its `requires_grad` is `False`: backpropagation skips it and
the optimiser never moves it. ResNet-18's parameters have names like `conv1.weight`,
`layer1.0.conv1.weight`, …, `layer4.1.bn2.bias`, `fc.weight`, `fc.bias`
(`model.named_parameters()` lists them).

**Task 2:** complete `make_model(train)`. It loads the trained network, replaces `fc` with a new
`nn.Linear(512, 4)`, and sets `requires_grad` so that only these parameters can move:

| `train` | trainable | name test |
|---|---|---|
| `"fc"` | the new last layer only (= feature extraction) | name starts with `"fc"` |
| `"layer4"` | the last stage + the new last layer | starts with `"layer4"` or `"fc"` |
| `"all"` | everything (= fine-tuning) | always |

(`name.startswith(("layer4", "fc"))` tests both prefixes at once.)

In [ ]:
def make_model(train="all"):
    '''A trained ResNet-18 with a new 4-class last layer; only the parts named by `train` can learn.

    train is one of: ("fc", "layer4", "all")

    "fc" = train only the last layer
    "layer4" = train the last stage + the last layer
    "all" = train everything
    '''
    model = pretrained_resnet18()
    # ── YOUR CODE HERE

    #remove this
    raise NotImplementedError("Task 2: new head, then freeze.")

    #replace the last layer
    #model.fc = nn.Linear(512, 4) # 1. new last layer: 512 -> 4

    #set-up a dictionary aligning the train argument with the layer names to train
    #prefixes = {"fc": ("fc",), "layer4": ("layer4", "fc"), "all": ("",)}

    #select the prefixes list based on the train arg
    #trainings = prefixes[train]

    # iterate over all the parameters in the network
    #for name, p in model.named_parameters():                   # 2-3. freeze everything else
        # set requires_grad based on the layers name (starts with)
    #    p.requires_grad = name.startswith(trainings)
    #return model.to(DEVICE)


In [ ]:
# Check Task 2: trainable weights for each choice. Do "fc" by hand first (Q1b).
try:
    for train in ("fc", "layer4", "all"):
        m = make_model(train)
        k, tot = n_params(m, trainable_only=True), n_params(m)
        print(f"train = {train!r:9s} {k:12,d} of {tot:,d} trainable ({k / tot:6.2%})")
    assert isinstance(m.fc, nn.Linear) and m.fc.out_features == 4, "fc should be a new nn.Linear(512, 4)"
    assert n_params(make_model("fc"), True) == 2052, "train='fc': only fc (2,052 weights) should be trainable"
    assert n_params(make_model("layer4"), True) == 8395780, "train='layer4': layer4 + fc should be trainable"
    assert n_params(make_model("all"), True) == n_params(m), "train='all': everything should be trainable"
    print("Task 2 OK. Output for 2 tiles:", tuple(m(prep(X_val[:2])).shape))
except NotImplementedError:
    print("Complete Tasks 1 and 2 first.")

### Step 2 · The fine-tuning loop (given)

`fine_tune` follows L14's recipe (slide 12):

* start from **Part 1's trained last layer**, not a random one, so the first gradients don't flow
  back into the good features as big, random corrections;
* **SGD, momentum 0.9, a small learning rate**, decaying linearly to 0 over the run (as in AL7 Part 3);
* keep batch norm's ImageNet averages: its layers stay in `eval` mode during training (`freeze_bn`);
* `augment` is `None` for now. Part 3 adds it.

After every epoch it measures accuracy on the training tiles (as they are, no augmentation) and on
`val`. Each epoch runs 11 million weights forwards and backwards on 630 tiles: a few seconds on a T4,
about 15 s on a CPU.

In [ ]:
EPOCHS, BATCH = 8, 32

def freeze_bn(model):
    for mod in model.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.eval()                                     # keep ImageNet's running averages

@torch.no_grad()
def predict_proba(model, tiles, bs=128):
    model.eval()
    return torch.cat([model(prep(tiles[i:i + bs])).softmax(1) for i in range(0, len(tiles), bs)]).cpu().numpy()

def accuracy(model, tiles, y):
    return (predict_proba(model, tiles).argmax(1) == y).mean()

def fine_tune(train="all", lr=0.0003, epochs=EPOCHS, augment=None, seed=0, label=""):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(train)
    model.fc.load_state_dict(HEAD.state_dict())            # start from Part 1's trained last layer
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9)
    steps = epochs * int(np.ceil(len(X_train) / BATCH))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1 - s / steps)   # lr -> 0 linearly
    Y = torch.as_tensor(y_train, device=DEVICE)
    hist = {"loss": [], "train_acc": [], "val_acc": []}
    t0 = time.time()
    for ep in range(epochs):
        model.train(); freeze_bn(model)
        perm = np.random.permutation(len(X_train)); total = 0.0
        for i in range(0, len(X_train), BATCH):
            k = perm[i:i + BATCH]
            xb = torch.as_tensor(X_train[k], device=DEVICE).float()      # (B, 64, 64), 0..255
            if augment is not None:
                xb = augment(xb)
            loss = F.cross_entropy(model(prep(xb)), Y[k])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item() * len(k)
            if not np.isfinite(loss.item()):
                break
        hist["loss"].append(total / len(X_train))
        hist["train_acc"].append(accuracy(model, X_train, y_train))
        hist["val_acc"].append(accuracy(model, X_val, y_val))
        print(f"{label:>16s} epoch {ep + 1}: loss {hist['loss'][-1]:.3f}  train {hist['train_acc'][-1]:5.1%}"
              f"  val {hist['val_acc'][-1]:5.1%}  ({time.time() - t0:.0f} s)")
    hist["model"] = model
    return hist

def plot_runs(runs, labels, colours, title, ref=None):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
    for lab, col in zip(labels, colours):
        h = runs[lab]; e = np.arange(1, len(h["loss"]) + 1)
        ax[0].plot(e, h["loss"], "o-", color=col, label=lab)
        ax[1].plot(e, h["val_acc"], "o-", color=col, label=f"{lab}: val")
        ax[1].plot(e, h["train_acc"], "--", color=col, alpha=0.7, label=f"{lab}: train")
    if ref is not None:
        ax[1].axhline(ref, color=GREY, ls=":", label=f"feature extraction: val {ref:.1%}")
    ax[0].set(xlabel="epoch", ylabel="training loss", title="loss")
    ax[1].set(xlabel="epoch", ylabel="accuracy", title="accuracy (dashed = train)", ylim=(0.2, 1.02))
    ax[1].legend(fontsize=8.5, loc="lower right"); ax[0].legend(fontsize=9)
    plt.suptitle(title); plt.tight_layout(); plt.show()

RUNS = {}

### Step 3 · 🔮 How big a step?

In AL7, **0.05** was the best learning rate for a network starting from random weights. Here the
weights start out *good*. **Predict first:** fine-tuning everything at lr 0.05 for two epochs: what
happens to validation accuracy, compared with feature extraction's? Then the same at **0.0003**.

In [ ]:
try:
    RUNS["lr 0.05"] = fine_tune("all", lr=0.05, epochs=2, label="lr 0.05")
    print(f"feature extraction was {RESULTS['feature extraction']['val']:.1%} on val")
except (NotImplementedError, NameError):
    print("Complete Tasks 1 and 2 first.")

In [ ]:
try:
    RUNS["no augmentation"] = fine_tune("all", lr=0.0003, label="lr 0.0003")
    plot_runs(RUNS, ["no augmentation"], [NAVY], f"Fine-tuning everything, lr 0.0003, {EPOCHS} epochs",
              ref=RESULTS["feature extraction"]["val"])
    RESULTS["fine-tuning"] = {"val": RUNS["no augmentation"]["val_acc"][-1]}
except (NotImplementedError, NameError):
    print("Complete Tasks 1 and 2 first.")

### Checkpoint 2

**Q2a.** What happened at lr 0.05? Why does a network that starts from **trained**
weights need a much smaller learning rate than AL7's network that started from random ones?
(L12 slide 14: what does a step that is too big do near a minimum?)

*Your answer:*

**Q2b.** Compare fine-tuning at lr 0.0003 with feature extraction on `val`. Then look
at the **training** accuracy (dashed). What does the gap between the dashed and solid lines tell you,
and what would you try next to shrink it? (AL7 Part 3.)

*Your answer:*

---
## Part 3 — Augmentation for air photos  `⏱ ~11 min`

### Step 1 · Changes that keep the label  (Task 3)

A **vertical** air photo has no "up": a woodlot rotated 90° or mirrored is still a woodlot, and a
village is still a village. So all **8 rotations and flips** are safe (L14 slide 18), and so are
modest **brightness and contrast** changes: 1958 prints and scans vary a lot from frame to frame
(L2). That is more than AL7 could use: there, upside-down photos would never occur at test time.

**Task 3:** complete `augment`. It gets a batch of tiles `x`, shape `(B, 64, 64)`, float, 0–255, on
`DEVICE`, and returns a new batch of the same shape:

1. rotate the whole batch by a random multiple of 90°: `torch.rot90(x, k, dims=(1, 2))` with
   `k = np.random.randint(4)`;
2. mirror a random half of the tiles left–right: `x.flip(2)` flips every tile; use a random mask
   and `torch.where` to pick, per tile, flipped or not (AL7 Task 3);
3. per tile, a random contrast `c` in [0.8, 1.2] and brightness `b` in [−20, 20]:
   `x = (x − 128) · c + 128 + b`, then `clamp(0, 255)`.

(`torch.empty(B, 1, 1, device=x.device).uniform_(lo, hi)` makes one random number per tile.)

In [ ]:
def augment(x):
    '''x: (B, 64, 64) float tiles, 0..255, on DEVICE -> a randomly rotated / flipped / re-lit copy.'''
    B = len(x)

    # --- uncomment the below code lines
    # rotate
    #x = torch.rot90(x, np.random.randint(4), dims=(1, 2))           # 1. 0, 90, 180 or 270 degrees

    # random flip
    #flip = torch.rand(B, device=x.device) < 0.5                     # 2. mirror half of them

    # apply the flip
    #x = torch.where(flip[:, None, None], x.flip(2), x)

    # each image gets random contrasts each 0.8 -> 1.2
    #c = torch.empty(B, 1, 1, device=x.device).uniform_(0.8, 1.2)    # 3. contrast ...
    # random brightness shift -20 -> 20
    #b = torch.empty(B, 1, 1, device=x.device).uniform_(-20, 20)     #    ... and brightness, per tile

    # augment the brightness and contrast
    # contrast (x - 128) * c + 128, brightness: + b
    #return ((x - 128) * c + 128 + b).clamp(0, 255)
    raise NotImplementedError("Task 3: rotate, flip, re-light.")

In [ ]:
# Check Task 3, then look before you train: one tile and 7 augmented copies of it.
try:
    torch.manual_seed(1); np.random.seed(1)
    xb = torch.as_tensor(X_train[:64], device=DEVICE).float()
    out = augment(xb)
    assert tuple(out.shape) == (64, 64, 64), f"shape should stay (64, 64, 64), got {tuple(out.shape)}"
    assert out.min() >= 0 and out.max() <= 255, "values should stay in 0..255 (clamp)"
    assert not torch.equal(out, xb), "augment returned its input unchanged"
    orients = set()
    for s in range(40):
        np.random.seed(s); torch.manual_seed(s)
        o = augment(torch.arange(4.0, device=DEVICE).view(1, 2, 2).repeat(8, 1, 1) * 60 + 30)
        orients |= {tuple(np.argsort(t.cpu().numpy().ravel())) for t in o}     # which corner is darkest, ...
    assert len(orients) == 8, f"over 40 batches a tile should appear in all 8 orientations; saw {len(orients)}"
    print("Task 3 OK")
    k = int(np.where(y_train == CLASSES.index("built"))[0][3])
    np.random.seed(3); torch.manual_seed(3)
    copies = [augment(torch.as_tensor(X_train[k:k + 1], device=DEVICE).float())[0].cpu() for _ in range(7)]
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
    axes[0].imshow(X_train[k], cmap="gray", vmin=0, vmax=255); axes[0].set_title(f"original ({CLASSES[y_train[k]]})")
    for ax, c in zip(axes[1:], copies):
        ax.imshow(c, cmap="gray", vmin=0, vmax=255); ax.set_title("augmented")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()
except NotImplementedError:
    print("Complete Task 3 first.")

### Step 2 · Same recipe, augmented

🔮 **Predict first.** With augmentation, will **training** accuracy at the end be higher or lower
than in Part 2? And validation accuracy?

In [ ]:
try:
    RUNS["augmented"] = fine_tune("all", lr=0.0003, augment=augment, label="augmented")
    plot_runs(RUNS, ["no augmentation", "augmented"], [NAVY, GOLD],
              f"Fine-tuning everything, lr 0.0003, {EPOCHS} epochs: with and without augmentation",
              ref=RESULTS["feature extraction"]["val"])
    for lab in ("no augmentation", "augmented"):
        h = RUNS[lab]
        print(f"{lab:<16} train {h['train_acc'][-1]:5.1%}   val {h['val_acc'][-1]:5.1%}   "
              f"gap {h['train_acc'][-1] - h['val_acc'][-1]:+6.1%}")
    RESULTS["fine-tuning + augmentation"] = {"val": RUNS["augmented"]["val_acc"][-1]}
except (NotImplementedError, KeyError):
    print("Complete Tasks 1-3 (and Part 2) first.")

### Checkpoint 3

**Q3a.** How did augmentation change the training accuracy, the validation accuracy and
the gap between them? Explain why lower training accuracy is expected here. (What does the network
see in epoch 5 with augmentation that it never saw without it?)

*Your answer:*

**Q3b.** Your project imagery. Name one augmentation that is safe for your data and one
that would **change** the label or create images that never occur. (Think: oblique photos with a
horizon; a "north-facing shoreline" class; colour 2020 images vs grey 1958 ones.)

*Your answer:*

---
## Part 4 — Kings County, once  `⏱ ~6 min`

Every choice so far was made on `val`. We now pick **one** model by its validation accuracy, and
measure it **once** on the 260 Kings County tiles: a county none of the training tiles came from.

In [ ]:
try:
    assert "fine-tuning + augmentation" in RESULTS, "run Parts 1-3 first"
    print("validation accuracy of each recipe:")
    for k, v in RESULTS.items():
        print(f"   {k:28s} {v['val']:6.1%}")
    CHOICE = max(reversed(list(RESULTS)), key=lambda k: RESULTS[k]["val"])   # ties -> the later recipe
    print("chosen:", CHOICE)
    if CHOICE == "feature extraction":
        with torch.no_grad():
            P_test = HEAD(F_test).softmax(1).cpu().numpy()
    else:
        final = RUNS["augmented" if "augment" in CHOICE else "no augmentation"]["model"]
        P_test = predict_proba(final, X_test)
    pred = P_test.argmax(1)
    print(f"\nKINGS COUNTY (test) accuracy: {np.mean(pred == y_test):.1%}   (260 tiles)")
    if MY_A2_VAL is not None:
        print("(compare with your A2 TileCNN's test accuracy, if you measured it)")
except (KeyError, NameError, AssertionError):
    print("Complete Parts 1-3 first.")

In [ ]:
if "P_test" in globals():
    CM = np.zeros((4, 4), int)
    np.add.at(CM, (y_test, pred), 1)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 1.1]})
    ax[0].imshow(CM, cmap="Blues")
    for i in range(4):
        for j in range(4):
            ax[0].text(j, i, CM[i, j], ha="center", va="center", fontsize=12,
                       color="white" if CM[i, j] > CM.max() / 2 else "black")
    ax[0].set_xticks(range(4), CLASSES); ax[0].set_yticks(range(4), CLASSES)
    ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); ax[0].set_title("Kings County: confusion matrix")
    recall = CM.diagonal() / CM.sum(1)
    ax[1].barh(range(4), recall, color=[RED if r < 0.6 else TEAL for r in recall])
    ax[1].set_yticks(range(4), CLASSES); ax[1].set_xlim(0, 1); ax[1].invert_yaxis()
    for i, r in enumerate(recall):
        ax[1].text(r + 0.01, i, f"{r:.0%} of {CM[i].sum()}", va="center")
    ax[1].set_title("per class: share of each true class found")
    plt.tight_layout(); plt.show()

In [ ]:
# The 10 most confident mistakes on Kings County.
if "P_test" in globals():
    wrong = np.where(pred != y_test)[0]
    worst = wrong[np.argsort(-P_test[wrong, pred[wrong]])][:10]
    fig, axes = plt.subplots(2, 5, figsize=(15, 6.8), gridspec_kw={"hspace": 0.3})
    for k, ax in zip(worst, axes.ravel()):
        ax.imshow(X_test[k], cmap="gray", vmin=0, vmax=255); ax.axis("off")
        ax.set_title(f"said {CLASSES[pred[k]]} ({P_test[k, pred[k]]:.0%})\nlabel {CLASSES[y_test[k]]}", fontsize=10, color=RED)
    plt.suptitle("Confidently wrong: the 10 Kings County mistakes with the highest predicted probability", y=0.99)
    plt.show()

### Checkpoint 4

**Q4a.** Which class is found least often, and what is it mistaken for? Look at the
confident mistakes: name one where you think the **label** is arguable, and one where the network is
simply wrong. Why is *edge* hard for any classifier, whatever its features? (A2's rule: edge =
field meets woodland, inside one 148 m tile.)

*Your answer:*

**Q4b.** Kings County accuracy is a few points below `val`. Give two reasons why. Which
of the two numbers would you put in your project report, and why?

*Your answer:*

---
## ★ Bonus — B1 · Test-time augmentation

If rotating a tile doesn't change its label, we can use that at **test** time too: ask the network
about all 8 rotations and flips of each tile and **average the 8 probability vectors**. It costs 8
forward passes per tile and no training. (We try it on `val` first; Kings is already spent.)

In [ ]:
@torch.no_grad()
def predict_tta(model, tiles):
    x = torch.as_tensor(tiles, device=DEVICE).float()
    P = 0
    for k in range(4):
        r = torch.rot90(x, k, dims=(1, 2))
        for v in (r, r.flip(2)):
            P = P + predict_proba(model, v)
    return P / 8

if "augmented" in RUNS:
    m = RUNS["augmented"]["model"]
    print(f"augmented model, val: one view {accuracy(m, X_val, y_val):.1%}   "
          f"8 views averaged {(predict_tta(m, X_val).argmax(1) == y_val).mean():.1%}")
    m0 = RUNS["no augmentation"]["model"]
    print(f"no-augmentation model, val: one view {accuracy(m0, X_val, y_val):.1%}   "
          f"8 views averaged {(predict_tta(m0, X_val).argmax(1) == y_val).mean():.1%}")

**B1.** Did averaging over 8 views help? Would you expect it to help more for the model
trained **with** augmentation or **without**? (Has the no-augmentation model ever seen a rotated
tile?) With only 157 validation tiles, how big a difference would you need before believing it?
(One tile is 0.6 points.)

*Your answer:*

---
## Reflection — End of Lab

**R1.** Write the recipe you would use for your project's classifier in five short
steps, from "label some tiles" to "report one number". For each step, name the part of today's lab
that it comes from.

*Your answer:*

**R2.** Feature extraction took seconds; fine-tuning took minutes. Your team has
labelled 40 tiles so far. Which would you run first, and what result would tell you it is worth
labelling more before doing anything else?

*Your answer:*

---
## ✏️ Try at home

Not graded.

**A. Fewer labels.** Keep only 10 training tiles per class (`np.where(y_train == c)[0][:10]`). Run
feature extraction, then fine-tuning with and without augmentation (you will need to change
`fine_tune` to take the tiles as arguments). Which recipe holds up best? (L14 slide 13 did this on
CIFAR.)

**B. Only the last stage.** `fine_tune("layer4")`: how close does it get to fine-tuning everything,
and how much faster is each epoch?

**C. 224 instead of 112.** Set `SIZE = 224` and re-run Parts 1-3 on a GPU. Is the gain worth four
times the compute?

**D. Your own tiles.** The A2 tile kit (linked from A2's last section) cuts and labels tiles from
your project's image set and saves them in this same `.npz` format. Point `FILE` at yours (with
your own class names) and run the whole lab on your data.